## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI. Run this command in your terminal:

```bash
az login
```

or

```bash
az login --use-device-code
```

# 👤 Simple Context Provider - Customer KYC Agent

This notebook demonstrates how to create a **Simple Context Provider** that manages customer profile information for KYC (Know Your Customer) compliance.

## Features Covered:
- Creating a custom `ContextProvider` class
- Extracting structured information from conversations
- Providing dynamic context to agents based on collected data
- Managing conversation state across multiple turns

### Industry Use Case: Customer KYC Profile Collection

Our context provider will help banking agents:
- Collect required customer identification information
- Track KYC verification status
- Enforce compliance rules (must collect info before providing services)
- Store customer profile data for the session

### ⚠️ Important Disclaimer ⚠️
> **This notebook is for educational purposes only. All customer information is simulated. In production, ensure compliance with data privacy regulations (GDPR, CCPA, etc.).**

### 🔍 How Context Providers Work

| Method | When Called | Purpose |
|--------|-------------|--------|
| `invoking()` | Before agent responds | Provide additional context/instructions |
| `invoked()` | After agent responds | Extract and store information from conversation |
| `serialize()` | When saving state | Persist data for thread continuation |

## Prerequisites

Before running this notebook, ensure you have:

1. **Microsoft Foundry Project**: With a deployed model (gpt-4o recommended)
2. **Authentication**: Azure CLI installed and authenticated
3. **Environment Variables** in root `.env` file:
   - `AI_FOUNDRY_PROJECT_ENDPOINT`
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME`

If you need to use a different tenant:
```bash
az login --tenant <tenant-id>
```

## Import Libraries

Import the required libraries for creating a simple context provider:

In [1]:
# Copyright (c) Microsoft. All rights reserved.
import os
import sys
from importlib.metadata import version
from pathlib import Path
from typing import Any

from agent_framework import (
    Agent,
    AgentSession,
    ContextProvider,
    Message,
    SessionContext,
    SupportsAgentRun,
    SupportsChatGetResponse,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from pydantic import BaseModel

assert version("agent-framework-core") == "1.17.0", "Select the pinned project kernel."
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements.in").is_file())
assert Path(sys.executable).resolve() == (repo_root / ".venv/Scripts/python.exe").resolve(), (
    "Select the repository .venv kernel."
)
load_dotenv(repo_root / ".env", override=False)

endpoint = (
    os.getenv("FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AZURE_AI_PROJECT_ENDPOINT")
)
model = os.getenv("FOUNDRY_MODEL") or os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME") or "gpt-4o"
if not endpoint or not model:
    raise ValueError("Set a Foundry project endpoint and model deployment name in the environment.")

print(f"Kernel: {sys.executable}\nPython: {sys.version.split()[0]}")
print("Project endpoint and model: configured (values hidden)")


Kernel: c:\src\agentic-ai-immersion\.venv\Scripts\python.exe
Python: 3.13.15
Project endpoint and model: configured (values hidden)


## Configuration 📋

Set up the configuration for Microsoft Foundry:

In [2]:
# Microsoft Foundry configuration (resolved in the setup cell)
PROJECT_ENDPOINT = endpoint
MODEL_DEPLOYMENT = model

print("Project endpoint and model: configured (values hidden)")


Project endpoint and model: configured (values hidden)


## Define Customer Profile Model 📝

Create a Pydantic model to represent customer KYC information that the context provider will collect and manage:

In [3]:
class CustomerProfile(BaseModel):
    """Customer KYC profile information."""
    full_name: str | None = None
    account_type: str | None = None  # e.g., "checking", "savings", "investment"
    annual_income: str | None = None  # income bracket
    employment_status: str | None = None  # e.g., "employed", "self-employed", "retired"

print("✅ CustomerProfile model defined")
print("   Fields: full_name, account_type, annual_income, employment_status")

✅ CustomerProfile model defined
   Fields: full_name, account_type, annual_income, employment_status


## Create the KYC Context Provider 🔍

This is the core of the notebook - a custom `ContextProvider` that:

1. **`invoking()`**: Before each agent call, provides instructions based on what customer info is still missing
2. **`invoked()`**: After each agent call, extracts customer information from the conversation
3. **`serialize()`**: Saves the customer profile for thread persistence

This pattern is useful for:
- KYC compliance workflows
- Customer onboarding
- Progressive data collection
- Personalized service delivery

In [4]:
class CustomerKYCProvider(ContextProvider):
    """Context provider that manages customer KYC profile collection.

    This provider:
    - Extracts customer information from conversations
    - Provides context instructions based on what info is missing
    - Enforces KYC compliance by requiring information before service
    """

    def __init__(
        self,
        chat_client: SupportsChatGetResponse,
        customer_profile: CustomerProfile | None = None,
        **kwargs: Any,
    ):
        """Initialize the KYC context provider.

        Args:
            chat_client: The chat client to use for extracting information
            customer_profile: Optional pre-populated customer profile
            **kwargs: Additional fields to populate the profile
        """
        super().__init__(source_id="customer-kyc")
        self._chat_client = chat_client

        if customer_profile:
            self.customer_profile = customer_profile
        elif kwargs:
            self.customer_profile = CustomerProfile.model_validate(kwargs)
        else:
            self.customer_profile = CustomerProfile()

    async def after_run(
        self,
        *,
        agent: SupportsAgentRun,
        session: AgentSession,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        """Extract customer information from messages after each agent call.

        Called AFTER the agent responds. Uses the chat client to extract
        structured customer information from the conversation.
        """
        needs_extraction = (
            self.customer_profile.full_name is None
            or self.customer_profile.account_type is None
            or self.customer_profile.annual_income is None
            or self.customer_profile.employment_status is None
        )

        # 1.17.0: read turn messages via get_messages(include_input=True).
        input_messages = context.get_messages(include_input=True)
        if needs_extraction and input_messages:
            try:
                result = await self._chat_client.get_response(
                    input_messages,
                    options={
                        "instructions": (
                            "Extract customer KYC information from the message if present. "
                            "Look for: full name, account type (checking/savings/investment), "
                            "annual income bracket, and employment status (employed/self-employed/retired). "
                            "Return nulls for any information not provided."
                        ),
                        "response_format": CustomerProfile,
                    },
                )

                # Update profile with extracted data (only fill in missing fields)
                if result.value and isinstance(result.value, CustomerProfile):
                    if self.customer_profile.full_name is None and result.value.full_name:
                        self.customer_profile.full_name = result.value.full_name
                    if self.customer_profile.account_type is None and result.value.account_type:
                        self.customer_profile.account_type = result.value.account_type
                    if self.customer_profile.annual_income is None and result.value.annual_income:
                        self.customer_profile.annual_income = result.value.annual_income
                    if self.customer_profile.employment_status is None and result.value.employment_status:
                        self.customer_profile.employment_status = result.value.employment_status
            except Exception:
                pass  # Failed to extract, continue without updating

    async def before_run(
        self,
        *,
        agent: SupportsAgentRun,
        session: AgentSession,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        """Provide customer profile context before each agent call.

        Called BEFORE the agent responds. Adds instructions based on what
        customer information is still missing.
        """
        instructions: list[str] = []

        if self.customer_profile.full_name is None:
            instructions.append(
                "The customer's name is not yet known. Politely ask for their full name "
                "before providing any account services."
            )
        else:
            instructions.append(f"The customer's name is {self.customer_profile.full_name}.")

        if self.customer_profile.account_type is None:
            instructions.append(
                "Ask what type of account they are interested in (checking, savings, or investment)."
            )
        else:
            instructions.append(f"The customer is interested in a {self.customer_profile.account_type} account.")

        if self.customer_profile.employment_status is None:
            instructions.append(
                "For KYC compliance, ask about their employment status (employed, self-employed, or retired)."
            )
        else:
            instructions.append(f"The customer's employment status is: {self.customer_profile.employment_status}.")

        if self.customer_profile.annual_income is None:
            instructions.append(
                "For account suitability, ask about their approximate annual income bracket."
            )
        else:
            instructions.append(f"The customer's income bracket is: {self.customer_profile.annual_income}.")

        if all([
            self.customer_profile.full_name,
            self.customer_profile.account_type,
            self.customer_profile.employment_status,
            self.customer_profile.annual_income,
        ]):
            instructions.append(
                "\nKYC profile is complete! You can now provide full banking services and recommendations."
            )

        # 1.17.0: add instructions with the provider's source id.
        context.extend_instructions(self.source_id, " ".join(instructions))

    def serialize(self) -> str:
        """Serialize the customer profile for thread persistence."""
        return self.customer_profile.model_dump_json()

    def get_profile_status(self) -> dict:
        """Get the current status of the customer profile."""
        return {
            "full_name": self.customer_profile.full_name or "❌ Not provided",
            "account_type": self.customer_profile.account_type or "❌ Not provided",
            "employment_status": self.customer_profile.employment_status or "❌ Not provided",
            "annual_income": self.customer_profile.annual_income or "❌ Not provided",
            "is_complete": all([
                self.customer_profile.full_name,
                self.customer_profile.account_type,
                self.customer_profile.employment_status,
                self.customer_profile.annual_income,
            ]),
        }

print("✅ CustomerKYCProvider class defined")


✅ CustomerKYCProvider class defined


## Define Agent Instructions 🏦

Create instructions for our banking KYC agent:

In [5]:
AGENT_INSTRUCTIONS = """
You are a professional Banking Customer Service Representative for Contoso Bank.

## Your Role:
- Help customers open new accounts and understand banking products
- Collect required KYC (Know Your Customer) information
- Provide personalized recommendations based on customer profile

## Guidelines:
1. **Be Professional**: Use a warm, professional tone
2. **Follow KYC Rules**: Collect required information before providing detailed services
3. **Address by Name**: Once you know the customer's name, use it in your responses
4. **Be Helpful**: Explain why information is needed when asked

## Required Disclaimers:
- This is for informational purposes only
- Actual account opening requires formal application and verification

## Response Style:
- Keep responses concise and friendly
- Ask one or two questions at a time, not all at once
- Acknowledge information as customers provide it
"""

print("📝 Agent Instructions Configured")
print(f"   Length: {len(AGENT_INSTRUCTIONS)} characters")

📝 Agent Instructions Configured
   Length: 892 characters


## Run the KYC Agent Demo 🚀

Now we'll demonstrate the context provider in action with a simulated customer conversation. Watch how the agent:

1. Initially asks for required information
2. Progressively collects customer profile data
3. Changes behavior once KYC is complete

In [6]:
async def run_kyc_agent_demo():
    """Run the KYC agent demonstration with context provider."""

    print("=" * 60)
    print("👤 Customer KYC Agent Demo")
    print("   Using Simple Context Provider")
    print("=" * 60 + "\n")

    with AzureCliCredential() as credential:
        chat_client = FoundryChatClient(
            project_endpoint=PROJECT_ENDPOINT,
            model=MODEL_DEPLOYMENT,
            credential=credential,
        )
        try:
            kyc_provider = CustomerKYCProvider(chat_client)
            agent = Agent(
                client=chat_client,
                name="kyc-banking-agent",
                instructions=AGENT_INSTRUCTIONS,
                context_providers=[kyc_provider],
            )

            print(f"✅ Agent created: {agent.name}")
            print("🔍 Context provider: CustomerKYCProvider\n")

            customer_messages = [
                "Hi, I'd like to open a new account please.",
                "My name is Sarah Johnson.",
                "I'm interested in a savings account.",
                "I work as a software engineer, so I'm employed full-time.",
                "My annual income is around $120,000.",
                "What savings accounts would you recommend for me?",
            ]

            # One session keeps the conversation coherent across turns.
            session = agent.create_session()
            for message in customer_messages:
                print("-" * 60)
                print(f"👤 Customer: {message}\n")

                import asyncio

                async with asyncio.timeout(90):
                    response = await agent.run(message, session=session)
                assert response.text, "The service returned no answer."
                print(f"🏦 Agent: {response.text}\n")

                status = kyc_provider.get_profile_status()
                print("📊 Profile Status:")
                print(f"   Name: {status['full_name']}")
                print(f"   Account Type: {status['account_type']}")
                print(f"   Employment: {status['employment_status']}")
                print(f"   Income: {status['annual_income']}")
                print(f"   KYC Complete: {'✅ Yes' if status['is_complete'] else '❌ No'}\n")

            print("=" * 60)
            print("✅ Demo complete!\n")
            print("📋 Final Customer Profile:")
            print(f"   {kyc_provider.serialize()}")
        finally:
            await chat_client.client.close()
            await chat_client.project_client.close()

# Run the demo
await run_kyc_agent_demo()


👤 Customer KYC Agent Demo
   Using Simple Context Provider

✅ Agent created: kyc-banking-agent
🔍 Context provider: CustomerKYCProvider

------------------------------------------------------------
👤 Customer: Hi, I'd like to open a new account please.

🏦 Agent: Thank you for your interest in opening an account with Contoso Bank! May I please have your full name to get started? Also, could you let me know what type of account you’re interested in—checking, savings, or investment?

📊 Profile Status:
   Name: ❌ Not provided
   Account Type: ❌ Not provided
   Employment: ❌ Not provided
   Income: ❌ Not provided
   KYC Complete: ❌ No

------------------------------------------------------------
👤 Customer: My name is Sarah Johnson.

🏦 Agent: Thank you, Sarah Johnson. Nice to meet you!

To help you choose the most suitable account, may I ask which type of account you’re interested in—checking, savings, or investment? Also, could you please share your current employment status (employed, self

## Interactive Mode 💬

Use this cell to have your own conversation with the KYC agent:

In [7]:
# Global variables to maintain state for interactive mode
_agent = None
_kyc_provider = None
_credential = None
_chat_client = None
_session = None

async def start_interactive_session():
    """Start an interactive KYC session."""
    global _agent, _kyc_provider, _credential, _chat_client, _session

    print("🚀 Starting interactive KYC session...\n")

    _credential = AzureCliCredential()
    _chat_client = FoundryChatClient(
        project_endpoint=PROJECT_ENDPOINT,
        model=MODEL_DEPLOYMENT,
        credential=_credential,
    )

    _kyc_provider = CustomerKYCProvider(_chat_client)

    _agent = Agent(
        client=_chat_client,
        name="interactive-kyc-agent",
        instructions=AGENT_INSTRUCTIONS,
        context_providers=[_kyc_provider],
    )
    _session = _agent.create_session()

    print("✅ Interactive session ready!")
    print("   Use ask_kyc_agent('your message') to chat")
    print("   Use show_profile_status() to see collected info")
    print("   Use end_interactive_session() when done\n")

async def ask_kyc_agent(message: str):
    """Send a message to the KYC agent."""
    if _agent is None:
        print("❌ Session not started. Run start_interactive_session() first.")
        return

    print(f"👤 You: {message}\n")
    response = await _agent.run(message, session=_session)
    print(f"🏦 Agent: {response.text}\n")

    status = _kyc_provider.get_profile_status()
    if status['is_complete']:
        print("✅ KYC Profile Complete!")
    else:
        missing = [k for k, v in status.items() if v == "❌ Not provided"]
        print(f"📊 Still needed: {', '.join(missing)}")

def show_profile_status():
    """Show the current customer profile status."""
    if _kyc_provider is None:
        print("❌ Session not started.")
        return

    status = _kyc_provider.get_profile_status()
    print("📋 Customer Profile Status:")
    print(f"   Name: {status['full_name']}")
    print(f"   Account Type: {status['account_type']}")
    print(f"   Employment: {status['employment_status']}")
    print(f"   Income: {status['annual_income']}")
    print(f"   KYC Complete: {'✅ Yes' if status['is_complete'] else '❌ No'}")

async def end_interactive_session():
    """End the interactive session and cleanup."""
    global _agent, _kyc_provider, _credential, _chat_client, _session

    if _chat_client is not None:
        await _chat_client.client.close()
        await _chat_client.project_client.close()
    if _credential is not None:
        _credential.close()

    _agent = None
    _kyc_provider = None
    _credential = None
    _chat_client = None
    _session = None

    print("✅ Interactive session ended.")

# Start the session
await start_interactive_session()


🚀 Starting interactive KYC session...

✅ Interactive session ready!
   Use ask_kyc_agent('your message') to chat
   Use show_profile_status() to see collected info
   Use end_interactive_session() when done



In [8]:
# Example interactive usage - uncomment and modify to try
await ask_kyc_agent("Hello, I want to open an account")
await ask_kyc_agent("My name is John Smith")
show_profile_status()

👤 You: Hello, I want to open an account

🏦 Agent: Thank you for your interest in opening an account with Contoso Bank! To assist you further, may I please have your full name?

Also, could you let me know which type of account you are interested in—checking, savings, or investment?

📊 Still needed: full_name, account_type, employment_status, annual_income
👤 You: My name is John Smith

🏦 Agent: Thank you, John Smith.

To help you choose the most suitable account, could you please tell me which type of account you are interested in: checking, savings, or investment?

Additionally, for regulatory purposes, may I ask about your employment status? Are you employed, self-employed, or retired?

📊 Still needed: account_type, employment_status, annual_income
📋 Customer Profile Status:
   Name: John Smith
   Account Type: ❌ Not provided
   Employment: ❌ Not provided
   Income: ❌ Not provided
   KYC Complete: ❌ No


In [9]:
# End the interactive session when done
await end_interactive_session()

✅ Interactive session ended.


## Key Takeaways 📚

### Creating a Simple Context Provider

```python
from agent_framework import ContextProvider, AgentSession, SessionContext, SupportsAgentRun
from pydantic import BaseModel

class MyDataModel(BaseModel):
    field1: str | None = None
    field2: str | None = None

class MyContextProvider(ContextProvider):
    def __init__(self, chat_client, **kwargs):
        super().__init__(source_id="my-provider")
        self._chat_client = chat_client
        self.data = MyDataModel()
    
    async def before_run(self, *, agent, session, context, state) -> None:
        # Called BEFORE agent responds
        # Add additional context/instructions to the session context
        context.extend_instructions(self.source_id, "Additional context for the agent...")
    
    async def after_run(self, *, agent, session, context, state) -> None:
        # Called AFTER agent responds
        # Extract and store information from conversation
        result = await self._chat_client.get_response(
            context.get_messages(include_input=True),
            options={"instructions": "Extract info...", "response_format": MyDataModel},
        )
        if result.value:
            self.data = result.value
    
    def serialize(self) -> str:
        return self.data.model_dump_json()
```

### Using the Context Provider with an Agent

```python
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient

# Create the chat client
chat_client = FoundryChatClient(
    project_endpoint=endpoint,
    model=model_deployment,
    credential=credential,
)

# Create provider
my_provider = MyContextProvider(chat_client)

# Create agent with provider
agent = Agent(
    client=chat_client,
    instructions="Your agent instructions",
    context_providers=[my_provider],
)
response = await agent.run("User message")
print(response.text)
```

### Context Provider Lifecycle

```
User Message → before_run() → Agent → after_run() → Response
                   ↓                       ↓
          Add instructions          Extract data
```

### Industry Use Cases for Context Providers

| Use Case | Data Collected | Compliance Benefit |
|----------|---------------|-------------------|
| KYC Verification | Name, ID, Address | Regulatory compliance |
| Risk Profiling | Income, Goals, Tolerance | Suitability requirements |
| Transaction Auth | Amount, Purpose, Recipient | AML compliance |
| Account Opening | Personal, Employment, Tax info | Documentation requirements |

### Migration Note

| Old (Deprecated) | New |
|---|---|
| `ContextProvider` | `ContextProvider` |
| `invoking(messages) → Context` | `before_run(*, agent, session, context, state) → None` |
| `invoked(request_messages, response_messages)` | `after_run(*, agent, session, context, state) → None` |
| `return Context(instructions=...)` | `context.instructions.append(...)` |
| `return Context(instructions=...)` | `context.extend_instructions(source_id, ...)` |

> Note: The Azure AI Search connector package in this environment still depends on an older BaseContextProvider API, so the search-based example may require a package version that matches the installed agent framework core.

### Environment Variables Needed

```env
# Required
AI_FOUNDRY_PROJECT_ENDPOINT=https://your-project.services.ai.azure.com/...
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4o
```

⚠️ **Disclaimer**: This notebook is for educational purposes. All customer scenarios are simulated. In production, ensure compliance with data privacy regulations (GDPR, CCPA, etc.) and banking regulations.